# 采购降本模拟演示

本 notebook 使用**全部虚构**的数据演示 2026 年 1—10 月的协议匹配、直接降价、三类替换、税率变化、异常和年度目标完成率。10 月是月报演示期；年度累计覆盖模拟生成器设定的 1—10 月。

数据不是 SRM 或 ERP 实际记录，示例中的供应商、物料、组织、采购员、返利和目标均为虚构；不读取、修改或归档业务数据库，也不写入钉钉。计算结果为**模拟初稿**，并非已复核的业务结果。

每个表变量保留完整数据，输出显示可能折叠；可用 Data Wrangler 查看完整明细。`明细`会在计算时重新生成，重跑会替换。金额示例单位为元，仅展示时保留两位；正式精度待确认。

In [1]:
from pathlib import Path
from decimal import Decimal
import pandas as pd
import sys
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:,.2f}".format)
数据目录 = Path("演示数据")
演示开始 = pd.Timestamp("2026-01-01")
演示结束 = pd.Timestamp("2026-10-31")
演示月份 = "2026-10"

# 各输入只读取一次；模拟入库的字段、顺序和编码类型对齐 processed 读取接口。
项目目录 = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/clean/procurement_inbound.py").exists())
sys.path.insert(0, str(项目目录 / "src"))
from clean.procurement_inbound import STANDARD_COLUMNS, UPDATED_AT_COLUMN
协议 = pd.read_csv(数据目录 / "模拟协议版本-扩展.csv", dtype={"协议编号": "string", "供应商编码": "string", "物料编码": "int64"}, parse_dates=["价格生效日期", "价格失效日期"])
入库 = pd.read_csv(数据目录 / "模拟采购入库-扩展.csv", dtype={"公司编码": "int64", "物料编码": "int64", "updated_at": "string"}, parse_dates=["出入库日期"])
替换 = pd.read_csv(数据目录 / "模拟替换填报-扩展.csv", dtype={"演示替换编号": "string", "适用采购主体": "int64", "原物料编码": "int64", "新物料编码": "int64"}, parse_dates=["价格生效日期"])
返利 = pd.read_csv(数据目录 / "模拟返利反馈-扩展.csv", dtype={"采购组织": "int64", "物料": "int64"}, parse_dates=["执行日期"])
assert 入库.columns.tolist() == STANDARD_COLUMNS + [UPDATED_AT_COLUMN], "模拟入库字段与 processed 不一致"
数量概览 = pd.DataFrame({"输入": ["协议版本", "采购入库", "替换填报", "返利反馈"], "行数": [len(协议), len(入库), len(替换), len(返利)]})
display(数量概览)
print("入库日期范围：", 入库["出入库日期"].min().date(), "至", 入库["出入库日期"].max().date())

,输入,行数
0,协议版本,448
1,采购入库,1971
2,替换填报,51
3,返利反馈,20


入库日期范围： 2026-01-15 至 2026-10-26


## 一、完整模拟输入

协议字段按已讨论的 SRM 导出清单构造；模拟入库与 processed 的 27 个标准字段及 `updated_at` 一致。后续计算中临时生成“演示行号”来定位模拟行，不属于真实来源字段，也不能证明真实来源已有稳定逐笔编号。模拟供应商名称可唯一对应编码；真实名称与编码仍需核对。

In [2]:
display(协议)
display(入库)
display(替换)
display(返利)

,协议编号,版本号,管理组织,采购模式,供应商编码,供应商名称,物料编码,产品名称,规格,含税价格,税率,价格生效日期,价格失效日期,适用组织编码
0,DEMO-A,1,演示总部,集中采购,S001,演示供应商甲,171529298,演示检测试剂,100测试/盒,100,0.13,2026-01-01,2026-01-31,ORG01
1,DEMO-A,2,演示总部,集中采购,S001,演示供应商甲,171529298,演示检测试剂,100测试/盒,90,0.13,2026-02-01,2026-06-30,ORG01
2,DEMO-B,1,演示总部,集中采购,S001,演示供应商甲,171529298,演示检测试剂,100测试/盒,80,0.13,2026-07-01,2027-06-30,ORG01
3,DEMO-C,1,演示总部,集中采购,S002,演示供应商乙,310594174,演示校准液,1瓶/瓶,50,0.13,2026-01-01,2026-09-30,ORG01
4,DEMO-C,2,演示总部,集中采购,S002,演示供应商乙,310594174,演示校准液,1瓶/瓶,50,0.09,2026-10-01,2027-09-30,ORG01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,MOCK-UP-A009,2,演示总部,集中采购,MOCK-US009,演示回升供应商09,475294461,演示产品MOCK-UP009,100测试/盒,90,0.13,2026-02-01,2026-09-30,DEMO-适用组织
444,MOCK-UP-A009,3,演示总部,集中采购,MOCK-US009,演示回升供应商09,475294461,演示产品MOCK-UP009,100测试/盒,95,0.13,2026-10-01,2027-09-30,DEMO-适用组织
445,MOCK-UP-A010,1,演示总部,集中采购,MOCK-US010,演示回升供应商10,298702733,演示产品MOCK-UP010,100测试/盒,100,0.13,2026-01-01,2026-01-31,DEMO-适用组织
446,MOCK-UP-A010,2,演示总部,集中采购,MOCK-US010,演示回升供应商10,298702733,演示产品MOCK-UP010,100测试/盒,90,0.13,2026-02-01,2026-09-30,DEMO-适用组织


,出入库日期,物料编码,物料名称,规格,主单位,一级分类,二级分类,三级分类,四级分类,五级分类,品牌,主品牌,货号,供应商名称,供应商类型,板块,公司编码,公司名称,公司简称,大区,数据来源系统,设备试剂分类,采购模式,数量,采购单价,采购金额(万),税率,updated_at
0,2026-10-08,171529298,演示检测试剂,100测试/盒,盒,演示一级,演示二级,演示三级,演示四级,演示甲类,演示品牌,演示主品牌,演示货号,演示供应商甲,外部单位,诊断,353402932,演示公司一,演示公司一,平台,nc,试剂/耗材,集中采购,100,80,0.80,0.13,2026-11-03T08:00:00+08:00
1,2026-10-18,171529298,演示检测试剂,100测试/盒,盒,演示一级,演示二级,演示三级,演示四级,演示甲类,演示品牌,演示主品牌,演示货号,演示供应商甲,外部单位,诊断,353402932,演示公司一,演示公司一,平台,nc,试剂/耗材,集中采购,-10,80,-0.08,0.13,2026-11-03T08:00:00+08:00
2,2026-10-20,171529298,演示检测试剂,100测试/盒,盒,演示一级,演示二级,演示三级,演示四级,演示甲类,演示品牌,演示主品牌,演示货号,演示供应商甲,外部单位,诊断,353402932,演示公司一,演示公司一,平台,nc,试剂/耗材,集中采购,20,90,0.18,0.13,2026-11-03T08:00:00+08:00
3,2026-10-12,310594174,演示校准液,1瓶/瓶,瓶,演示一级,演示二级,演示三级,演示四级,演示乙类,演示品牌,演示主品牌,演示货号,演示供应商乙,外部单位,诊断,353402932,演示公司一,演示公司一,平台,nc,试剂/耗材,集中采购,40,50,0.20,0.09,2026-11-03T08:00:00+08:00
4,2026-10-15,282876899,演示多协议试剂,1盒/盒,盒,演示一级,演示二级,演示三级,演示四级,演示丙类,演示品牌,演示主品牌,演示货号,演示供应商丙,外部单位,诊断,353402932,演示公司一,演示公司一,平台,nc,试剂/耗材,集中采购,25,70,0.17,0.13,2026-11-03T08:00:00+08:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1966,2026-10-13,211251521,演示产品MOCK-SP011-NEW,演示规格,盒,演示一级,演示二级,演示三级,演示四级,演示替换类,演示品牌,演示主品牌,演示货号,演示新供应商SP011,外部单位,诊断,780628938,演示公司4,演示公司4,平台,nc,试剂/耗材,集中采购,20,50,0.10,0.13,2026-11-03T08:00:00+08:00
1967,2026-10-13,184248323,演示产品MOCK-SP012-NEW,演示规格,盒,演示一级,演示二级,演示三级,演示四级,演示替换类,演示品牌,演示主品牌,演示货号,演示新供应商SP012,外部单位,诊断,119824329,演示公司1,演示公司1,平台,nc,试剂/耗材,集中采购,11,50,0.06,0.13,2026-11-03T08:00:00+08:00
1968,2026-10-13,127424842,演示产品MOCK-SP013-NEW,演示规格,盒,演示一级,演示二级,演示三级,演示四级,演示替换类,演示品牌,演示主品牌,演示货号,演示新供应商SP013,外部单位,诊断,787431546,演示公司2,演示公司2,平台,nc,试剂/耗材,集中采购,16,50,0.08,0.13,2026-11-03T08:00:00+08:00
1969,2026-10-13,198774460,演示产品MOCK-SP014-NEW,演示规格,盒,演示一级,演示二级,演示三级,演示四级,演示替换类,演示品牌,演示主品牌,演示货号,演示新供应商SP014,外部单位,诊断,712329798,演示公司3,演示公司3,平台,nc,试剂/耗材,集中采购,23,50,0.12,0.13,2026-11-03T08:00:00+08:00


,演示替换编号,替换类型,适用采购主体,原供应商,新供应商,原物料编码,新物料编码,原含税价,现含税价,价格生效日期,原计价单位,新计价单位,可比单位,原换算系数,新换算系数,业务依据,复核状态
0,DEMO-R01,渠道替换,353402932,演示旧渠道,演示新渠道,454222210,454222210,100,90,2026-08-01,盒,盒,盒,1,1,演示填报：渠道替换,已确认
1,DEMO-R02,品牌替换,353402932,演示旧品牌,演示新品牌,246201696,317119447,100,45,2026-08-01,盒,盒,测试,100,50,演示填报：品牌替换,已确认
2,DEMO-R03,规格替换,626550946,演示旧规格,演示新规格,657728185,651503972,120,50,2026-08-01,盒,盒,测试,100,50,演示填报：规格替换,已确认
3,DEMO-R04,渠道替换,353402932,演示旧跨类型渠道,演示跨类型供应商,460886090,460886090,100,90,2026-07-01,盒,盒,盒,1,1,演示填报：渠道替换后直接降价,已确认
4,DEMO-R05,品牌替换,353402932,演示旧跨品牌供应商,演示跨品牌供应商,130039353,600772442,100,45,2026-07-01,盒,盒,测试,100,50,演示填报：品牌替换后直接降价,已确认
5,DEMO-R06,规格替换,626550946,演示旧跨规格供应商,演示跨规格供应商,258515483,441444215,120,50,2026-07-01,盒,盒,测试,100,50,演示填报：规格替换后直接降价,已确认
6,MOCK-R-CH001,渠道替换,787431546,演示旧供应商CH001,演示新供应商CH001,826351029,826351029,100,90,2026-08-01,盒,盒,盒,1,1,演示用业务依据,已确认
7,MOCK-R-CH002,渠道替换,712329798,演示旧供应商CH002,演示新供应商CH002,397604771,397604771,100,90,2026-08-01,盒,盒,盒,1,1,演示用业务依据,已确认
8,MOCK-R-CH003,渠道替换,780628938,演示旧供应商CH003,演示新供应商CH003,473481583,473481583,100,90,2026-08-01,盒,盒,盒,1,1,演示用业务依据,已确认
9,MOCK-R-CH004,渠道替换,119824329,演示旧供应商CH004,演示新供应商CH004,184595354,184595354,100,90,2026-08-01,盒,盒,盒,1,1,演示用业务依据,已确认


,采购组织,供应商,物料,五级分类,执行日期,返利比例,返利条款
0,712329798,演示供应商06,216202872,演示类别1,2026-10-01,0.03,演示返利条款，仅供反馈
1,119824329,演示供应商12,311218836,演示类别2,2026-10-01,0.04,演示返利条款，仅供反馈
2,712329798,演示供应商06,559121460,演示类别3,2026-10-01,0.02,演示返利条款，仅供反馈
3,119824329,演示供应商12,409951335,演示类别4,2026-10-01,0.03,演示返利条款，仅供反馈
4,712329798,演示供应商06,642697504,演示类别5,2026-10-01,0.04,演示返利条款，仅供反馈
5,119824329,演示供应商12,766639862,演示类别1,2026-10-01,0.02,演示返利条款，仅供反馈
6,712329798,演示供应商06,515475994,演示类别2,2026-10-01,0.03,演示返利条款，仅供反馈
7,119824329,演示供应商12,743783485,演示类别3,2026-10-01,0.04,演示返利条款，仅供反馈
8,712329798,演示供应商06,895941963,演示类别4,2026-10-01,0.02,演示返利条款，仅供反馈
9,119824329,演示供应商12,172564205,演示类别5,2026-10-01,0.03,演示返利条款，仅供反馈


### 10 月入库聚合

按已确认的“物料－供应商－单价”业务键聚合 10 月带符号数量。这里不加入适用组织；原始入库明细仍保留日期和采购主体，便于后续按月及主体核对。

In [3]:
本月入库 = 入库[入库["出入库日期"].dt.strftime("%Y-%m").eq(演示月份)].copy()
入库聚合 = 本月入库.groupby(["物料编码", "供应商名称", "采购单价"], dropna=False).agg(净入库数量=("数量", "sum"), 流水行数=("数量", "size")).reset_index()
print("10 月入库流水：", len(本月入库), "行；按物料、供应商、单价聚合：", len(入库聚合), "组")
display(入库聚合)

10 月入库流水： 1071 行；按物料、供应商、单价聚合： 200 组


,物料编码,供应商名称,采购单价,净入库数量,流水行数
0,106137739,演示供应商11,106,227,8
1,106444413,演示供应商01,132,229,8
2,108373331,演示新供应商SP007,50,12,1
3,121861544,演示新供应商BR005,45,9,1
4,121894033,演示新供应商CH011,90,7,1
...,...,...,...,...,...
195,886884234,演示供应商08,114,216,9
196,892376172,演示供应商06,136,214,8
197,895941963,演示供应商06,124,228,8
198,896677702,演示供应商03,111,219,8


## 二、识别前序协议

非 V1 查本协议上一版本；V1 及示例中的 V2 新增物料查其他协议，每份旧协议先取最后版本。按管理组织、采购模式、供应商和物料寻找候选，适用组织不参与连续性判断；时间重叠或间隔小于一个日历年才满足条件。

模拟中前序生效日必须早于当前生效日；V2 以外的上一版本缺失先标记待确认。2 月 29 日失效且不重叠的特殊边界不自动判断。找不到前序的模拟首版仅作为已构造的演示基准；真实业务中必须核对是否遗漏历史。

In [4]:
协议键 = ["协议编号", "版本号", "物料编码"]
assert not 协议.duplicated(协议键).any(), "演示协议键存在重复，先检查模拟数据"
协议 = 协议.sort_values(["价格生效日期", "协议编号", "版本号", "物料编码"]).reset_index(drop=True)
协议["演示协议明细键"] = 协议["协议编号"].astype(str) + "|V" + 协议["版本号"].astype(str) + "|" + 协议["物料编码"].astype(str)
协议索引 = 协议.set_index("演示协议明细键", drop=False)
旧协议最后版本 = 协议[协议["版本号"].eq(协议.groupby("协议编号")["版本号"].transform("max"))]

def 时间连续(旧, 新):
    if pd.isna(旧["价格失效日期"]) or pd.isna(新["价格生效日期"]):
        return None
    if 新["价格生效日期"] <= 旧["价格失效日期"]:
        return True
    if 旧["价格失效日期"].month == 2 and 旧["价格失效日期"].day == 29:
        return None  # 用户尚未确认 2 月 29 日的对日边界
    return 新["价格生效日期"] < 旧["价格失效日期"] + pd.DateOffset(years=1)

def 跨协议候选(新):
    c = 旧协议最后版本[
        (旧协议最后版本["协议编号"] != 新["协议编号"]) &
        (旧协议最后版本["管理组织"] == 新["管理组织"]) &
        (旧协议最后版本["采购模式"] == 新["采购模式"]) &
        (旧协议最后版本["供应商编码"] == 新["供应商编码"]) &
        (旧协议最后版本["物料编码"] == 新["物料编码"]) &
        (旧协议最后版本["价格生效日期"] < 新["价格生效日期"])
    ].copy()
    c["时间判断"] = [时间连续(旧, 新) for _, 旧 in c.iterrows()]
    return c

关系行, 候选行 = [], []
for _, 新 in 协议.iterrows():
    前序, 状态, 方式 = None, "", ""
    if 新["版本号"] > 1:
        上版 = 协议[(协议["协议编号"] == 新["协议编号"]) & (协议["版本号"] == 新["版本号"] - 1) & (协议["物料编码"] == 新["物料编码"])]
        if len(上版) == 1:
            方式 = "本协议上一版本"
            候选 = 上版.iloc[0]
            连续 = 时间连续(候选, 新)
            状态 = "已识别" if 连续 is True else "不连续" if 连续 is False else "时间边界待确认"
            前序 = 候选["演示协议明细键"] if 连续 is True else None
        elif 新["版本号"] != 2:
            状态, 方式 = "上一版本明细缺失待确认", "本协议上一版本"
    if not 状态:
        方式 = "跨协议末版本"
        c = 跨协议候选(新)
        for _, 旧 in c.iterrows():
            候选行.append({"当前明细": 新["演示协议明细键"], "候选旧明细": 旧["演示协议明细键"], "旧价格": 旧["含税价格"], "旧失效日期": 旧["价格失效日期"], "新生效日期": 新["价格生效日期"], "时间判断": 旧["时间判断"]})
        不明 = c[c["时间判断"].isna()]
        合格 = c[c["时间判断"].eq(True)]
        if len(不明):
            状态 = "时间边界待确认"
        elif len(合格) > 1:
            状态 = "多个上游待复核"
        elif len(合格) == 1:
            状态, 前序 = "已识别", 合格.iloc[0]["演示协议明细键"]
        else:
            状态 = "无前序（演示基准待核实）"
    关系行.append({"当前明细": 新["演示协议明细键"], "前序明细": 前序, "查找方式": 方式, "关系状态": 状态})

关系 = pd.DataFrame(关系行)
前序候选 = pd.DataFrame(候选行)
display(关系.groupby(["查找方式", "关系状态"], dropna=False).size().rename("明细数").reset_index())
display(关系)
display(前序候选)

,查找方式,关系状态,明细数
0,本协议上一版本,已识别,147
1,跨协议末版本,多个上游待复核,13
2,跨协议末版本,已识别,122
3,跨协议末版本,无前序（演示基准待核实）,166


,当前明细,前序明细,查找方式,关系状态
0,DEMO-G-OLD|V1|409943508,NaN,跨协议末版本,无前序（演示基准待核实）
1,MOCK-A0006|V1|216202872,NaN,跨协议末版本,无前序（演示基准待核实）
2,MOCK-A0012|V1|311218836,NaN,跨协议末版本,无前序（演示基准待核实）
3,MOCK-A0018|V1|559121460,NaN,跨协议末版本,无前序（演示基准待核实）
4,MOCK-A0024|V1|409951335,NaN,跨协议末版本,无前序（演示基准待核实）
...,...,...,...,...
443,MOCK-UP-A006|V3|312777055,MOCK-UP-A006|V2|312777055,本协议上一版本,已识别
444,MOCK-UP-A007|V3|205689588,MOCK-UP-A007|V2|205689588,本协议上一版本,已识别
445,MOCK-UP-A008|V3|411080767,MOCK-UP-A008|V2|411080767,本协议上一版本,已识别
446,MOCK-UP-A009|V3|475294461,MOCK-UP-A009|V2|475294461,本协议上一版本,已识别


,当前明细,候选旧明细,旧价格,旧失效日期,新生效日期,时间判断
0,DEMO-B|V1|171529298,DEMO-A|V2|171529298,90,2026-06-30,2026-07-01,True
1,MOCK-B0002|V1|546417262,MOCK-A0002|V2|546417262,112,2026-08-31,2026-07-01,True
2,MOCK-B0003|V1|745730165,MOCK-A0003|V2|745730165,113,2026-06-30,2026-07-01,True
3,MOCK-B0005|V1|206498114,MOCK-A0005|V2|206498114,115,2026-08-31,2026-07-01,True
4,MOCK-B0006|V1|216202872,MOCK-A0006|V2|216202872,116,2026-06-30,2026-07-01,True
...,...,...,...,...,...,...
144,MOCK-D3-010|V1|800206608,MOCK-D2-010|V1|800206608,95,2026-09-30,2026-10-01,True
145,MOCK-D3-011|V1|416850916,MOCK-D1-011|V1|416850916,100,2026-08-31,2026-10-01,True
146,MOCK-D3-011|V1|416850916,MOCK-D2-011|V1|416850916,95,2026-09-30,2026-10-01,True
147,MOCK-D3-012|V1|263140199,MOCK-D1-012|V1|263140199,100,2026-08-31,2026-10-01,True


## 三、税率变化独立展示

复用同一前序关系，只有前后税率不同才进入展示表；无法确认前序的记录不推定发生变化。展示新旧协议编号、版本及日期，不计算税率影响金额。税率按小数存储，例如 0.13 表示 13%。

In [5]:
关联索引 = 关系.set_index("当前明细")
税率变化行 = []
for _, 关系记录 in 关系[关系["关系状态"].eq("已识别")].iterrows():
    新 = 协议索引.loc[关系记录["当前明细"]]
    旧 = 协议索引.loc[关系记录["前序明细"]]
    if 新["税率"] != 旧["税率"]:
        税率变化行.append({"供应商编码": 新["供应商编码"], "供应商名称": 新["供应商名称"], "物料编码": 新["物料编码"], "产品名称": 新["产品名称"], "规格": 新["规格"], "管理组织": 新["管理组织"], "采购模式": 新["采购模式"], "旧协议编号": 旧["协议编号"], "旧版本号": 旧["版本号"], "原税率": 旧["税率"], "旧价格生效日期": 旧["价格生效日期"], "旧价格失效日期": 旧["价格失效日期"], "新协议编号": 新["协议编号"], "新版本号": 新["版本号"], "新税率": 新["税率"], "新价格生效日期": 新["价格生效日期"], "新价格失效日期": 新["价格失效日期"]})
税率变化 = pd.DataFrame(税率变化行)
print("税率变化记录：", len(税率变化))
display(税率变化)

税率变化记录： 13


,供应商编码,供应商名称,物料编码,产品名称,规格,管理组织,采购模式,旧协议编号,旧版本号,原税率,旧价格生效日期,旧价格失效日期,新协议编号,新版本号,新税率,新价格生效日期,新价格失效日期
0,MOCK-S08,演示供应商08,675220764,演示产品MOCK-M0020,100测试/盒,演示管理组织2,自行采购,MOCK-A0020,2,0.13,2026-02-01,2026-08-31,MOCK-B0020,1,0.09,2026-07-01,2027-07-31
1,MOCK-S06,演示供应商06,642697504,演示产品MOCK-M0030,100测试/盒,演示管理组织3,集中采购,MOCK-A0030,2,0.13,2025-12-01,2026-06-30,MOCK-B0030,1,0.09,2026-07-01,2027-07-31
2,MOCK-S02,演示供应商02,338448831,演示产品MOCK-M0050,100测试/盒,演示管理组织2,自行采购,MOCK-A0050,2,0.13,2026-02-01,2026-08-31,MOCK-B0050,1,0.09,2026-07-01,2027-07-31
3,MOCK-S12,演示供应商12,172564205,演示产品MOCK-M0060,100测试/盒,演示管理组织3,集中采购,MOCK-A0060,2,0.13,2025-12-01,2026-06-30,MOCK-B0060,1,0.09,2026-07-01,2027-07-31
4,MOCK-S08,演示供应商08,539474118,演示产品MOCK-M0080,100测试/盒,演示管理组织2,自行采购,MOCK-A0080,2,0.13,2026-02-01,2026-08-31,MOCK-B0080,1,0.09,2026-07-01,2027-07-31
5,MOCK-S06,演示供应商06,219918641,演示产品MOCK-M0090,100测试/盒,演示管理组织3,集中采购,MOCK-A0090,2,0.13,2025-12-01,2026-06-30,MOCK-B0090,1,0.09,2026-07-01,2027-07-31
6,MOCK-S02,演示供应商02,802734415,演示产品MOCK-M0110,100测试/盒,演示管理组织2,自行采购,MOCK-A0110,2,0.13,2026-02-01,2026-08-31,MOCK-B0110,1,0.09,2026-07-01,2027-07-31
7,MOCK-S12,演示供应商12,854060767,演示产品MOCK-M0120,100测试/盒,演示管理组织3,集中采购,MOCK-A0120,2,0.13,2025-12-01,2026-06-30,MOCK-B0120,1,0.09,2026-07-01,2027-07-31
8,MOCK-S10,演示供应商10,574872217,演示产品MOCK-M0010,100测试/盒,演示管理组织1,定向采购,MOCK-A0010,2,0.13,2026-02-01,2026-05-31,MOCK-B0010,1,0.09,2026-08-01,2027-07-31
9,MOCK-S04,演示供应商04,231797130,演示产品MOCK-M0040,100测试/盒,演示管理组织1,定向采购,MOCK-A0040,2,0.13,2026-02-01,2026-05-31,MOCK-B0040,1,0.09,2026-08-01,2027-07-31


## 四、匹配入库并计算模拟降本

匹配业务键为物料、供应商、含税单价；入库日期和采购主体保留，适用组织不加入此键。每个降价事项独立核算 12 个月：1 日生效当月开始，否则下月开始。负数量按发生日期冲减。

三类替换使用模拟填报的关系及换算系数；替换后的直接降价可以通过已确认替换关系与对应的协议价格链连接，分别计算各自的价差。存在多个候选或不能区分价差时先列为待复核，不自动重复归因。协议候选不唯一、未匹配价格、价格回升等也列为待复核。

In [6]:
def 周期(生效日):
    开始 = 生效日.replace(day=1) if 生效日.day == 1 else (生效日 + pd.offsets.MonthBegin(1)).normalize()
    return 开始, 开始 + pd.DateOffset(months=12) - pd.Timedelta(days=1)

def D(value):
    return Decimal(str(value))

def 明细行(入库行, 类型, 来源, 原价, 现价, 单位差, 核算量, 单位, 开始, 结束, 原计价单位=None, 新计价单位=None):
    return {"核算月份": 入库行["出入库日期"].strftime("%Y-%m"), "出入库日期": 入库行["出入库日期"], "公司编码": 入库行["公司编码"], "公司名称": 入库行["公司名称"], "供应商名称": 入库行["供应商名称"], "物料编码": 入库行["物料编码"], "物料名称": 入库行["物料名称"], "五级分类": 入库行["五级分类"], "演示行号": 入库行["演示行号"], "核算类型": 类型, "降本来源": 来源, "原含税价": float(原价), "原计价单位": 原计价单位 or 单位, "现含税价": float(现价), "新计价单位": 新计价单位 or 单位, "实际入库数量": 入库行["数量"], "核算数量": float(核算量), "核算单位": 单位, "单位降本": float(单位差), "降本金额": float(单位差 * 核算量), "核算开始": 开始, "核算结束": 结束, "新增存量": "新增" if 入库行["出入库日期"].year == 开始.year else "存量"}

计算行, 异常行 = [], []
分析入库 = 入库[入库["出入库日期"].between(演示开始, 演示结束)].copy()
分析入库["演示行号"] = 分析入库.index + 1  # 仅本次演示内的临时行位置
for _, r in 分析入库.iterrows():
    替换候选 = 替换[(替换["复核状态"] == "已确认") & (替换["新供应商"] == r["供应商名称"]) & (替换["新物料编码"] == r["物料编码"]) & (替换["现含税价"] == r["采购单价"]) & (替换["适用采购主体"] == r["公司编码"])].copy()
    替换候选 = 替换候选[[周期(d)[0] <= r["出入库日期"] <= 周期(d)[1] for d in 替换候选["价格生效日期"]]]
    价格候选 = 协议[(协议["供应商名称"] == r["供应商名称"]) & (协议["物料编码"] == r["物料编码"]) & (协议["含税价格"] == r["采购单价"]) & (协议["价格生效日期"] <= r["出入库日期"]) & (协议["价格失效日期"] >= r["出入库日期"])]
    if len(替换候选) > 1 or len(价格候选) > 1 or (len(替换候选) and len(价格候选)):
        异常行.append({"演示行号": r["演示行号"], "原因": "多个匹配或直接与替换可能重复归因", "物料编码": r["物料编码"], "供应商名称": r["供应商名称"], "采购单价": r["采购单价"], "数量": r["数量"]})
        continue
    if len(替换候选) == 1:
        t = 替换候选.iloc[0]
        if D(t["原换算系数"]) <= 0 or D(t["新换算系数"]) <= 0:
            异常行.append({"演示行号": r["演示行号"], "原因": "换算系数异常", "物料编码": r["物料编码"], "供应商名称": r["供应商名称"], "采购单价": r["采购单价"], "数量": r["数量"]})
            continue
        差 = D(t["原含税价"]) / D(t["原换算系数"]) - D(t["现含税价"]) / D(t["新换算系数"])
        if 差 < 0:
            异常行.append({"演示行号": r["演示行号"], "原因": "替换成本增加待复核", "物料编码": r["物料编码"], "供应商名称": r["供应商名称"], "采购单价": r["采购单价"], "数量": r["数量"]})
            continue
        开始, 结束 = 周期(t["价格生效日期"])
        计算行.append(明细行(r, t["替换类型"], t["演示替换编号"], t["原含税价"], t["现含税价"], 差, D(r["数量"]) * D(t["新换算系数"]), t["可比单位"], 开始, 结束, t["原计价单位"], t["新计价单位"]))
        continue
    if len(价格候选) == 0:
        异常行.append({"演示行号": r["演示行号"], "原因": "未匹配有效协议价格或替换关系", "物料编码": r["物料编码"], "供应商名称": r["供应商名称"], "采购单价": r["采购单价"], "数量": r["数量"]})
        continue
    当前 = 价格候选.iloc[0]
    当前键 = 当前["演示协议明细键"]
    入库当前键 = 当前键
    路径, 待复核原因, 已遍历 = [], None, set()
    while 当前键:
        if 当前键 in 已遍历:
            待复核原因 = "协议关联成环待复核"
            break
        已遍历.add(当前键)
        关系记录 = 关联索引.loc[当前键]
        if 关系记录["关系状态"] == "无前序（演示基准待核实）":
            if 当前键 == 入库当前键:
                待复核原因 = "当前价格无可确认的前序价格"
            break
        if 关系记录["关系状态"] != "已识别":
            待复核原因 = 关系记录["关系状态"]
            break
        当前项 = 协议索引.loc[当前键]
        前项 = 协议索引.loc[关系记录["前序明细"]]
        if D(当前项["含税价格"]) > D(前项["含税价格"]):
            待复核原因 = "价格回升待采购员确认"
            break
        if D(当前项["含税价格"]) < D(前项["含税价格"]):
            路径.append((当前项, 前项))
        当前键 = 关系记录["前序明细"]
    if 待复核原因:
        异常行.append({"演示行号": r["演示行号"], "原因": 待复核原因, "物料编码": r["物料编码"], "供应商名称": r["供应商名称"], "采购单价": r["采购单价"], "数量": r["数量"]})
        continue
    # 以采购员已确认的替换关系连接协议链上的执行价格，避免把不同价差当作重复。
    历史执行价格 = {协议索引.loc[k, "含税价格"] for k in 已遍历}
    跨类型替换 = 替换[(替换["复核状态"] == "已确认") & (替换["新供应商"] == r["供应商名称"]) & (替换["新物料编码"] == r["物料编码"]) & (替换["现含税价"].isin(历史执行价格)) & (替换["适用采购主体"] == r["公司编码"])].copy()
    跨类型替换 = 跨类型替换[[周期(d)[0] <= r["出入库日期"] <= 周期(d)[1] for d in 跨类型替换["价格生效日期"]]]
    if len(跨类型替换) > 1:
        异常行.append({"演示行号": r["演示行号"], "原因": "多个跨类型替换关系待复核", "物料编码": r["物料编码"], "供应商名称": r["供应商名称"], "采购单价": r["采购单价"], "数量": r["数量"]})
        continue
    if len(跨类型替换) == 1:
        t = 跨类型替换.iloc[0]
        if D(t["原换算系数"]) <= 0 or D(t["新换算系数"]) <= 0:
            异常行.append({"演示行号": r["演示行号"], "原因": "换算系数异常", "物料编码": r["物料编码"], "供应商名称": r["供应商名称"], "采购单价": r["采购单价"], "数量": r["数量"]})
            continue
        差 = D(t["原含税价"]) / D(t["原换算系数"]) - D(t["现含税价"]) / D(t["新换算系数"])
        if 差 < 0:
            异常行.append({"演示行号": r["演示行号"], "原因": "替换成本增加待复核", "物料编码": r["物料编码"], "供应商名称": r["供应商名称"], "采购单价": r["采购单价"], "数量": r["数量"]})
            continue
        开始, 结束 = 周期(t["价格生效日期"])
        计算行.append(明细行(r, t["替换类型"], t["演示替换编号"], t["原含税价"], t["现含税价"], 差, D(r["数量"]) * D(t["新换算系数"]), t["可比单位"], 开始, 结束, t["原计价单位"], t["新计价单位"]))
    for 当前项, 前项 in 路径:
        开始, 结束 = 周期(当前项["价格生效日期"])
        if 开始 <= r["出入库日期"] <= 结束:
            差 = D(前项["含税价格"]) - D(当前项["含税价格"])
            计算行.append(明细行(r, "直接降价", 当前项["演示协议明细键"], 前项["含税价格"], 当前项["含税价格"], 差, D(r["数量"]), r["主单位"], 开始, 结束))

明细 = pd.DataFrame(计算行)
异常 = pd.DataFrame(异常行)
print("分析入库行数：", len(分析入库), "；计算明细行数：", len(明细), "；待复核入库行数：", len(异常))
display(明细)
display(异常.groupby("原因", dropna=False).size().rename("入库行数").reset_index())
display(异常)

分析入库行数： 1971 ；计算明细行数： 3259 ；待复核入库行数： 26


,核算月份,出入库日期,公司编码,公司名称,供应商名称,物料编码,物料名称,五级分类,演示行号,核算类型,降本来源,原含税价,原计价单位,现含税价,新计价单位,实际入库数量,核算数量,核算单位,单位降本,降本金额,核算开始,核算结束,新增存量
0,2026-10,2026-10-08,353402932,演示公司一,演示供应商甲,171529298,演示检测试剂,演示甲类,1,直接降价,DEMO-B|V1|171529298,90.00,盒,80.00,盒,100,100.00,盒,10.00,"1,000.00",2026-07-01,2027-06-30,新增
1,2026-10,2026-10-08,353402932,演示公司一,演示供应商甲,171529298,演示检测试剂,演示甲类,1,直接降价,DEMO-A|V2|171529298,100.00,盒,90.00,盒,100,100.00,盒,10.00,"1,000.00",2026-02-01,2027-01-31,新增
2,2026-10,2026-10-18,353402932,演示公司一,演示供应商甲,171529298,演示检测试剂,演示甲类,2,直接降价,DEMO-B|V1|171529298,90.00,盒,80.00,盒,-10,-10.00,盒,10.00,-100.00,2026-07-01,2027-06-30,新增
3,2026-10,2026-10-18,353402932,演示公司一,演示供应商甲,171529298,演示检测试剂,演示甲类,2,直接降价,DEMO-A|V2|171529298,100.00,盒,90.00,盒,-10,-10.00,盒,10.00,-100.00,2026-02-01,2027-01-31,新增
4,2026-10,2026-10-13,353402932,演示公司一,演示新渠道,454222210,演示渠道物料,演示替换类,7,渠道替换,DEMO-R01,100.00,盒,90.00,盒,50,50.00,盒,10.00,500.00,2026-08-01,2027-07-31,新增
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3254,2026-10,2026-10-13,780628938,演示公司4,演示新供应商SP011,211251521,演示产品MOCK-SP011-NEW,演示替换类,1967,规格替换,MOCK-R-SP011,120.00,盒,50.00,盒,20,"1,000.00",测试,0.20,200.00,2026-08-01,2027-07-31,新增
3255,2026-10,2026-10-13,119824329,演示公司1,演示新供应商SP012,184248323,演示产品MOCK-SP012-NEW,演示替换类,1968,规格替换,MOCK-R-SP012,120.00,盒,50.00,盒,11,550.00,测试,0.20,110.00,2026-08-01,2027-07-31,新增
3256,2026-10,2026-10-13,787431546,演示公司2,演示新供应商SP013,127424842,演示产品MOCK-SP013-NEW,演示替换类,1969,规格替换,MOCK-R-SP013,120.00,盒,50.00,盒,16,800.00,测试,0.20,160.00,2026-08-01,2027-07-31,新增
3257,2026-10,2026-10-13,712329798,演示公司3,演示新供应商SP014,198774460,演示产品MOCK-SP014-NEW,演示替换类,1970,规格替换,MOCK-R-SP014,120.00,盒,50.00,盒,23,"1,150.00",测试,0.20,230.00,2026-08-01,2027-07-31,新增


,原因,入库行数
0,价格回升待采购员确认,11
1,多个上游待复核,13
2,当前价格无可确认的前序价格,1
3,未匹配有效协议价格或替换关系,1


,演示行号,原因,物料编码,供应商名称,采购单价,数量
0,3,未匹配有效协议价格或替换关系,171529298,演示供应商甲,90,20
1,5,多个上游待复核,282876899,演示供应商丙,70,25
2,6,价格回升待采购员确认,486923143,演示供应商丁,95,10
3,11,当前价格无可确认的前序价格,409943508,演示供应商己,80,20
4,1905,多个上游待复核,815876748,演示多协议供应商01,70,20
5,1906,多个上游待复核,669658224,演示多协议供应商02,70,20
6,1907,多个上游待复核,434847200,演示多协议供应商03,70,20
7,1908,多个上游待复核,580129042,演示多协议供应商04,70,20
8,1909,多个上游待复核,788872660,演示多协议供应商05,70,20
9,1910,多个上游待复核,451772804,演示多协议供应商06,70,20


## 五、月报与模拟个人目标

月报以 2026 年 10 月为例，年度目标完成率仅在本套**明确由生成器构造的模拟年度数据**上演示。采购员和目标是虚构参数，金额是模拟初稿；未匹配和待复核记录不计入。真实业务必须先核对来源完整性和业务复核，不得将缺档自动补零。

In [7]:
演示负责人规则 = {f"演示类别{i}": f"演示采购员{i}" for i in range(1, 6)}
演示负责人规则.update({"演示甲类": "演示采购员1", "演示替换类": "演示采购员6"})
明细["采购负责人"] = 明细["五级分类"].map(演示负责人规则).fillna("待确认")
月度明细 = 明细[明细["核算月份"] == 演示月份].copy()
月度汇总 = 月度明细.groupby(["核算类型", "新增存量"], dropna=False)["降本金额"].sum().reset_index()
月度维度 = 月度明细.groupby(["公司名称", "采购负责人", "五级分类", "供应商名称", "核算类型", "新增存量"], dropna=False)["降本金额"].sum().reset_index()
目标 = pd.DataFrame({"采购负责人": [f"演示采购员{i}" for i in range(1, 7)], "年度目标金额": [150000.0] * 6})
年度累计 = 明细.groupby("采购负责人", dropna=False)["降本金额"].sum().rename("年度累计降本").reset_index()
完成率 = 目标.merge(年度累计, on="采购负责人", how="left")
完成率["年度累计降本"] = 完成率["年度累计降本"].fillna(0)  # 仅本模拟生成器声明的完整演示范围
完成率["目标完成率"] = 完成率["年度累计降本"] / 完成率["年度目标金额"]
print("10 月模拟降本合计（元）：", f"{月度明细['降本金额'].sum():,.2f}")
display(月度汇总)
display(月度维度)
display(完成率.assign(**{"目标完成率(%)": 完成率["目标完成率"] * 100}).drop(columns="目标完成率"))
print("完整月度明细：", len(月度明细), "行；重跑会替换变量 明细。")
display(月度明细)

10 月模拟降本合计（元）： 407,785.00


,核算类型,新增存量,降本金额
0,品牌替换,新增,"1,310.00"
1,渠道替换,新增,"3,490.00"
2,直接降价,存量,"44,530.00"
3,直接降价,新增,"355,725.00"
4,规格替换,新增,"2,730.00"


,公司名称,采购负责人,五级分类,供应商名称,核算类型,新增存量,降本金额
0,演示公司1,演示采购员1,演示类别1,演示供应商01,直接降价,新增,"1,950.00"
1,演示公司1,演示采购员1,演示类别1,演示供应商02,直接降价,新增,"1,125.00"
2,演示公司1,演示采购员1,演示类别1,演示供应商03,直接降价,新增,"2,100.00"
3,演示公司1,演示采购员1,演示类别1,演示供应商04,直接降价,新增,"1,920.00"
4,演示公司1,演示采购员1,演示类别1,演示供应商05,直接降价,新增,"2,070.00"
...,...,...,...,...,...,...,...
331,演示公司一,演示采购员6,演示替换类,演示跨类型供应商,渠道替换,新增,"1,000.00"
332,演示公司一,演示采购员6,演示替换类,演示跨类型供应商,直接降价,新增,"1,000.00"
333,演示公司二,演示采购员6,演示替换类,演示新规格,规格替换,新增,100.00
334,演示公司二,演示采购员6,演示替换类,演示跨规格供应商,直接降价,新增,50.00


,采购负责人,年度目标金额,年度累计降本,目标完成率(%)
0,演示采购员1,"150,000.00","142,470.00",94.98
1,演示采购员2,"150,000.00","138,700.00",92.47
2,演示采购员3,"150,000.00","139,860.00",93.24
3,演示采购员4,"150,000.00","138,725.00",92.48
4,演示采购员5,"150,000.00","138,590.00",92.39
5,演示采购员6,"150,000.00","8,680.00",5.79


完整月度明细： 2039 行；重跑会替换变量 明细。


,核算月份,出入库日期,公司编码,公司名称,供应商名称,物料编码,物料名称,五级分类,演示行号,核算类型,降本来源,原含税价,原计价单位,现含税价,新计价单位,实际入库数量,核算数量,核算单位,单位降本,降本金额,核算开始,核算结束,新增存量,采购负责人
0,2026-10,2026-10-08,353402932,演示公司一,演示供应商甲,171529298,演示检测试剂,演示甲类,1,直接降价,DEMO-B|V1|171529298,90.00,盒,80.00,盒,100,100.00,盒,10.00,"1,000.00",2026-07-01,2027-06-30,新增,演示采购员1
1,2026-10,2026-10-08,353402932,演示公司一,演示供应商甲,171529298,演示检测试剂,演示甲类,1,直接降价,DEMO-A|V2|171529298,100.00,盒,90.00,盒,100,100.00,盒,10.00,"1,000.00",2026-02-01,2027-01-31,新增,演示采购员1
2,2026-10,2026-10-18,353402932,演示公司一,演示供应商甲,171529298,演示检测试剂,演示甲类,2,直接降价,DEMO-B|V1|171529298,90.00,盒,80.00,盒,-10,-10.00,盒,10.00,-100.00,2026-07-01,2027-06-30,新增,演示采购员1
3,2026-10,2026-10-18,353402932,演示公司一,演示供应商甲,171529298,演示检测试剂,演示甲类,2,直接降价,DEMO-A|V2|171529298,100.00,盒,90.00,盒,-10,-10.00,盒,10.00,-100.00,2026-02-01,2027-01-31,新增,演示采购员1
4,2026-10,2026-10-13,353402932,演示公司一,演示新渠道,454222210,演示渠道物料,演示替换类,7,渠道替换,DEMO-R01,100.00,盒,90.00,盒,50,50.00,盒,10.00,500.00,2026-08-01,2027-07-31,新增,演示采购员6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3254,2026-10,2026-10-13,780628938,演示公司4,演示新供应商SP011,211251521,演示产品MOCK-SP011-NEW,演示替换类,1967,规格替换,MOCK-R-SP011,120.00,盒,50.00,盒,20,"1,000.00",测试,0.20,200.00,2026-08-01,2027-07-31,新增,演示采购员6
3255,2026-10,2026-10-13,119824329,演示公司1,演示新供应商SP012,184248323,演示产品MOCK-SP012-NEW,演示替换类,1968,规格替换,MOCK-R-SP012,120.00,盒,50.00,盒,11,550.00,测试,0.20,110.00,2026-08-01,2027-07-31,新增,演示采购员6
3256,2026-10,2026-10-13,787431546,演示公司2,演示新供应商SP013,127424842,演示产品MOCK-SP013-NEW,演示替换类,1969,规格替换,MOCK-R-SP013,120.00,盒,50.00,盒,16,800.00,测试,0.20,160.00,2026-08-01,2027-07-31,新增,演示采购员6
3257,2026-10,2026-10-13,712329798,演示公司3,演示新供应商SP014,198774460,演示产品MOCK-SP014-NEW,演示替换类,1970,规格替换,MOCK-R-SP014,120.00,盒,50.00,盒,23,"1,150.00",测试,0.20,230.00,2026-08-01,2027-07-31,新增,演示采购员6


## 六、管理层与采购员的模拟分析结果

以下是对前面已生成的 dataframe 再汇总，不重新取数。10 月统计以 `核算月份` 为准；负数入库按发生月冲减。年度目标和人员均为虚构示例，数据覆盖仅限生成器构造的模拟期间。金额单位为元，结果为待复核的模拟初稿。

In [8]:
# 管理层：每类降本的新增、存量及合计，单位元。
类型报表 = 月度明细.groupby(["核算类型", "新增存量"], dropna=False)["降本金额"].sum().unstack(fill_value=0).reindex(columns=["新增", "存量"], fill_value=0)
类型报表["合计"] = 类型报表["新增"] + 类型报表["存量"]
类型报表 = 类型报表.reindex(["直接降价", "渠道替换", "品牌替换", "规格替换"])
类型报表.loc["合计"] = 类型报表.sum()
assert abs(类型报表.loc["合计", "合计"] - 月度明细["降本金额"].sum()) < 1e-8
print("10 月按类型和新增／存量汇总（元）")
display(类型报表)

10 月按类型和新增／存量汇总（元）


新增存量,新增,存量,合计
核算类型,,,
直接降价,"355,725.00","44,530.00","400,255.00"
渠道替换,"3,490.00",0.00,"3,490.00"
品牌替换,"1,310.00",0.00,"1,310.00"
规格替换,"2,730.00",0.00,"2,730.00"
合计,"363,255.00","44,530.00","407,785.00"


In [9]:
# 按主体、五级分类和供应商分别汇总；保留全部维度值供核对。
主体报表 = 月度明细.groupby(["公司编码", "公司名称"], dropna=False)["降本金额"].sum().reset_index().sort_values("降本金额", ascending=False)
分类报表 = 月度明细.groupby("五级分类", dropna=False)["降本金额"].sum().reset_index().sort_values("降本金额", ascending=False)
供应商报表 = 月度明细.groupby("供应商名称", dropna=False)["降本金额"].sum().reset_index().sort_values("降本金额", ascending=False)
assert all(abs(t["降本金额"].sum() - 月度明细["降本金额"].sum()) < 1e-8 for t in [主体报表, 分类报表, 供应商报表])
print("按采购主体（元）")
display(主体报表)
print("按五级分类（元）")
display(分类报表)
print("按供应商（元，完整明细）")
display(供应商报表)

按采购主体（元）


,公司编码,公司名称,降本金额
0,119824329,演示公司1,"105,330.00"
5,787431546,演示公司2,"99,680.00"
4,780628938,演示公司4,"99,550.00"
3,712329798,演示公司3,"98,075.00"
1,353402932,演示公司一,"4,900.00"
2,626550946,演示公司二,250.00


按五级分类（元）


,五级分类,降本金额
6,演示类别5,"82,890.00"
5,演示类别4,"79,395.00"
3,演示类别2,"78,555.00"
2,演示类别1,"78,525.00"
4,演示类别3,"77,640.00"
0,演示替换类,"8,680.00"
1,演示甲类,"2,100.00"


按供应商（元，完整明细）


,供应商名称,降本金额
7,演示供应商08,"35,055.00"
1,演示供应商02,"34,410.00"
0,演示供应商01,"34,365.00"
11,演示供应商12,"33,945.00"
4,演示供应商05,"33,285.00"
9,演示供应商10,"33,285.00"
10,演示供应商11,"33,255.00"
5,演示供应商06,"32,850.00"
3,演示供应商04,"32,235.00"
6,演示供应商07,"32,070.00"


In [10]:
# 采购员：10 月、年度累计、年度目标及完成率。
负责人月度 = 月度明细.groupby("采购负责人", dropna=False)["降本金额"].sum().rename("10月降本").reset_index()
采购员报表 = 完成率.merge(负责人月度, on="采购负责人", how="left")
采购员报表["10月降本"] = 采购员报表["10月降本"].fillna(0)  # 模拟范围由生成器完整构造
采购员报表 = 采购员报表[["采购负责人", "10月降本", "年度累计降本", "年度目标金额", "目标完成率"]]
采购员报表["目标完成率(%)"] = 采购员报表.pop("目标完成率") * 100
采购员报表 = 采购员报表.sort_values("10月降本", ascending=False).reset_index(drop=True)
print("采购员目标完成情况：目标及负责人均为虚构")
display(采购员报表)

采购员目标完成情况：目标及负责人均为虚构


,采购负责人,10月降本,年度累计降本,年度目标金额,目标完成率(%)
0,演示采购员5,"82,890.00","138,590.00","150,000.00",92.39
1,演示采购员1,"80,625.00","142,470.00","150,000.00",94.98
2,演示采购员4,"79,395.00","138,725.00","150,000.00",92.48
3,演示采购员2,"78,555.00","138,700.00","150,000.00",92.47
4,演示采购员3,"77,640.00","139,860.00","150,000.00",93.24
5,演示采购员6,"8,680.00","8,680.00","150,000.00",5.79


In [11]:
# 待复核：仅使用模拟输入中的行位置关联日期，不能作为真实来源主键。
定位入库 = 入库.reset_index(names="模拟行索引").copy()
定位入库["演示行号"] = 定位入库["模拟行索引"] + 1
异常带日期 = 异常.merge(定位入库[["演示行号", "出入库日期", "公司名称"]], on="演示行号", how="left", validate="one_to_one")
本月异常 = 异常带日期[异常带日期["出入库日期"].dt.strftime("%Y-%m").eq(演示月份)].copy()
异常原因汇总 = 本月异常.groupby("原因", dropna=False).size().rename("笔数").reset_index()
本月税率变化 = 税率变化[税率变化["新价格生效日期"].dt.strftime("%Y-%m").eq(演示月份)].copy()
print("10 月待复核入库：", len(本月异常), "笔")
display(异常原因汇总)
print("全部待复核明细：")
display(本月异常)
print("协议版本中税率变化共", len(税率变化), "条，其中 10 月生效", len(本月税率变化), "条")
display(本月税率变化)

10 月待复核入库： 26 笔


,原因,笔数
0,价格回升待采购员确认,11
1,多个上游待复核,13
2,当前价格无可确认的前序价格,1
3,未匹配有效协议价格或替换关系,1


全部待复核明细：


,演示行号,原因,物料编码,供应商名称,采购单价,数量,出入库日期,公司名称
0,3,未匹配有效协议价格或替换关系,171529298,演示供应商甲,90,20,2026-10-20,演示公司一
1,5,多个上游待复核,282876899,演示供应商丙,70,25,2026-10-15,演示公司一
2,6,价格回升待采购员确认,486923143,演示供应商丁,95,10,2026-10-16,演示公司一
3,11,当前价格无可确认的前序价格,409943508,演示供应商己,80,20,2026-10-19,演示公司一
4,1905,多个上游待复核,815876748,演示多协议供应商01,70,20,2026-10-15,演示公司1
5,1906,多个上游待复核,669658224,演示多协议供应商02,70,20,2026-10-15,演示公司1
6,1907,多个上游待复核,434847200,演示多协议供应商03,70,20,2026-10-15,演示公司1
7,1908,多个上游待复核,580129042,演示多协议供应商04,70,20,2026-10-15,演示公司1
8,1909,多个上游待复核,788872660,演示多协议供应商05,70,20,2026-10-15,演示公司1
9,1910,多个上游待复核,451772804,演示多协议供应商06,70,20,2026-10-15,演示公司1


协议版本中税率变化共 13 条，其中 10 月生效 1 条


,供应商编码,供应商名称,物料编码,产品名称,规格,管理组织,采购模式,旧协议编号,旧版本号,原税率,旧价格生效日期,旧价格失效日期,新协议编号,新版本号,新税率,新价格生效日期,新价格失效日期
12,S002,演示供应商乙,310594174,演示校准液,1瓶/瓶,演示总部,集中采购,DEMO-C,1,0.13,2026-01-01,2026-09-30,DEMO-C,2,0.09,2026-10-01,2027-09-30


## 七、演示核对

这些断言仅核对虚构样例中的已知场景，不能证明真实来源覆盖、唯一性和业务口径已经验收。

In [12]:
assert len(协议) == 448 and len(入库) == 1971 and len(替换) == 51 and len(返利) == 20
assert 入库聚合["净入库数量"].sum() == 本月入库["数量"].sum()
def 示例协议键(协议编号, 版本号):
    命中 = 协议[(协议["协议编号"] == 协议编号) & (协议["版本号"] == 版本号)]
    assert len(命中) == 1
    return 命中.iloc[0]["演示协议明细键"]
assert 关联索引.loc[示例协议键("DEMO-B", 1), "前序明细"] == 示例协议键("DEMO-A", 2)
assert 关联索引.loc[示例协议键("DEMO-D3", 1), "关系状态"] == "多个上游待复核"
assert set(前序候选.loc[前序候选["当前明细"] == 示例协议键("DEMO-D3", 1), "候选旧明细"]) == {示例协议键("DEMO-D1", 1), 示例协议键("DEMO-D2", 1)}
assert len(税率变化) == 13
assert len(税率变化[税率变化["新协议编号"] == "DEMO-C"]) == 1
assert 关联索引.loc[示例协议键("DEMO-F-NEW", 2), "前序明细"] == 示例协议键("DEMO-F-OLD", 1)
assert 明细.loc[明细["演示行号"] == 1, "降本金额"].sum() == 2000
assert 明细.loc[明细["演示行号"] == 2, "降本金额"].sum() == -200
assert 明细.loc[明细["演示行号"] == 7, "降本金额"].sum() == 500
assert 明细.loc[明细["演示行号"] == 8, "降本金额"].sum() == 100
assert 明细.loc[明细["演示行号"] == 9, "降本金额"].sum() == 100
assert 明细.loc[明细["演示行号"] == 10, "降本金额"].sum() == 300
跨类型 = 明细[明细["演示行号"] == 12]
assert set(跨类型["核算类型"]) == {"渠道替换", "直接降价"} and 跨类型["降本金额"].sum() == 2000
跨品牌 = 明细[明细["演示行号"] == 13]
assert set(跨品牌["核算类型"]) == {"品牌替换", "直接降价"} and 跨品牌["降本金额"].sum() == 200
跨规格 = 明细[明细["演示行号"] == 14]
assert set(跨规格["核算类型"]) == {"规格替换", "直接降价"} and 跨规格["降本金额"].sum() == 150
assert {5, 6, 11}.issubset(set(异常["演示行号"]))
assert set(月度汇总["核算类型"]) == {"直接降价", "渠道替换", "品牌替换", "规格替换"}
print("模拟场景核对通过；真实数据、业务口径及正式归档仍待确认。")

模拟场景核对通过；真实数据、业务口径及正式归档仍待确认。


## 八、生成模拟看板快照

将本 notebook 已计算的模拟结果写入独立快照，供采购增量同一个本地服务展示。快照只包含虚构数据；输入文件改变后需重跑 notebook。

In [13]:
import hashlib, json
from datetime import datetime
from zoneinfo import ZoneInfo

输入文件 = ["模拟协议版本-扩展.csv", "模拟采购入库-扩展.csv", "模拟替换填报-扩展.csv", "模拟返利反馈-扩展.csv"]
def 打包(frame):
    return json.loads(frame.to_json(orient="records", date_format="iso", date_unit="s", force_ascii=False))

模拟入库 = 入库.copy()
模拟入库.insert(0, "演示行号", 模拟入库.index + 1)
异常展示 = 异常.merge(模拟入库[["演示行号", "出入库日期", "公司编码", "公司名称", "五级分类"]], on="演示行号", how="left", validate="one_to_one")
异常展示["采购负责人"] = 异常展示["五级分类"].map(演示负责人规则).fillna("待确认")
快照 = {
    "meta": {"schema_version": 1, "data_kind": "mock", "生成时间": datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(timespec="seconds"),
             "业务开始": str(演示开始.date()), "业务结束": str(演示结束.date()),
             "输入校验": {name: hashlib.sha256((数据目录 / name).read_bytes()).hexdigest() for name in 输入文件}},
    "detail": 打包(明细), "receipts": 打包(模拟入库), "exceptions": 打包(异常展示),
    "tax_changes": 打包(税率变化), "rebates": 打包(返利), "targets": 打包(目标),
    "protocols": 打包(协议), "relations": 打包(关系), "candidates": 打包(前序候选)
}
快照路径 = 数据目录 / "模拟降本看板快照.json"
快照路径.write_text(json.dumps(快照, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
print("已生成模拟看板快照：", 快照路径, "；核算明细", len(明细), "行")

已生成模拟看板快照： 演示数据/模拟降本看板快照.json ；核算明细 3259 行
